# Phase 4 — Spirale 2D, frontière non-linéaire visualisée

## Objectif

Générer un dataset en spirale, entraîner un réseau dessus, puis visualiser la frontière de décision.

Dans cette phase, on va :
- générer une spirale 2D ;
- entraîner un réseau 2–64–64–1 ;
- suivre la loss et l'accuracy ;
- exporter la figure finale.

In [20]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [21]:
def generate_spiral(n_points=400, noise=0.15, seed=42):
    """Génère deux spirales entrelacées : classe 0 et classe 1."""
    np.random.seed(seed)
    n = n_points // 2

    theta0 = np.linspace(0, 4 * np.pi, n) + np.random.randn(n) * noise
    theta1 = np.linspace(0, 4 * np.pi, n) + np.random.randn(n) * noise + np.pi

    r = np.linspace(0.1, 1.0, n)

    X0 = np.c_[r * np.cos(theta0), r * np.sin(theta0)]
    X1 = np.c_[r * np.cos(theta1), r * np.sin(theta1)]

    X = np.vstack([X0, X1])
    y = np.hstack([np.zeros(n), np.ones(n)])
    return X, y

In [22]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def tanh_grad(a):
    return 1 - a**2

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

## 1. Génération des données

On génère une spirale avec :
- `n_points = 400`
- `noise = 0.15`

In [23]:
X, y = generate_spiral(n_points=400, noise=0.15, seed=42)

print("Shape X :", X.shape)
print("Shape y :", y.shape)
print("Classes :", np.unique(y))

Shape X : (400, 2)
Shape y : (400,)
Classes : [0. 1.]


In [24]:
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", s=12)
plt.title("Dataset spirale 2D")
plt.xlabel("x1")
plt.ylabel("x2")
plt.grid(True)
plt.show()

C:\Users\serge\AppData\Local\Temp\ipykernel_13208\2870861911.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Initialisation du réseau

Architecture :
- 2 entrées ;
- 64 neurones cachés ;
- 64 neurones cachés ;
- 1 sortie.

On utilise une initialisation He adaptée à ReLU :
\(\sqrt{2 / n_{in}}\)

In [25]:
def train_spiral(seed, X, y, hidden_dim=32, learning_rate=0.1, n_epochs=8000):
    np.random.seed(seed)

    W1 = np.random.randn(2, hidden_dim) * 0.5
    b1 = np.zeros((1, hidden_dim))

    W2 = np.random.randn(hidden_dim, hidden_dim) * 0.5
    b2 = np.zeros((1, hidden_dim))

    W3 = np.random.randn(hidden_dim, 1) * 0.5
    b3 = np.zeros((1, 1))

    losses = []
    n = len(X)

    for epoch in range(n_epochs):
        # Forward
        z1 = X @ W1 + b1
        a1 = tanh(z1)

        z2 = a1 @ W2 + b2
        a2 = tanh(z2)

        z3 = a2 @ W3 + b3
        y_pred = sigmoid(z3).flatten()

        # Loss
        loss = bce_loss(y, y_pred)
        losses.append(loss)

        # Backward
        err3 = (y_pred - y).reshape(-1, 1)
        dW3 = (a2.T @ err3) / n
        db3 = np.sum(err3, axis=0, keepdims=True) / n

        err2 = (err3 @ W3.T) * tanh_grad(a2)
        dW2 = (a1.T @ err2) / n
        db2 = np.sum(err2, axis=0, keepdims=True) / n

        err1 = (err2 @ W2.T) * tanh_grad(a1)
        dW1 = (X.T @ err1) / n
        db1 = np.sum(err1, axis=0, keepdims=True) / n

        # Update
        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    final_pred = y_pred
    final_acc = np.mean((final_pred > 0.5) == y)
    final_loss = losses[-1]

    return {
        "seed": seed,
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2,
        "W3": W3, "b3": b3,
        "losses": losses,
        "y_pred": final_pred,
        "acc": final_acc,
        "loss": final_loss
    }

## 3. Entraînement multi-seeds

On entraîne plusieurs modèles et on garde celui qui obtient :
1. la meilleure accuracy ;
2. puis la plus faible loss en cas d'égalité.

In [26]:
results = []

for seed in range(10):
    result = train_spiral(
        seed=seed,
        X=X,
        y=y,
        hidden_dim=32,
        learning_rate=0.1,
        n_epochs=8000
    )
    results.append(result)
    print(f"Seed {seed:2d} | Loss: {result['loss']:.4f} | Accuracy: {result['acc']:.2%}")

Seed  0 | Loss: 0.1576 | Accuracy: 97.75%
Seed  1 | Loss: 0.0506 | Accuracy: 100.00%
Seed  2 | Loss: 0.1006 | Accuracy: 99.75%
Seed  3 | Loss: 0.0848 | Accuracy: 99.00%
Seed  4 | Loss: 0.0968 | Accuracy: 99.75%
Seed  5 | Loss: 0.1143 | Accuracy: 98.75%
Seed  6 | Loss: 0.1051 | Accuracy: 99.25%
Seed  7 | Loss: 0.0825 | Accuracy: 99.75%
Seed  8 | Loss: 0.0932 | Accuracy: 99.25%
Seed  9 | Loss: 0.0902 | Accuracy: 99.75%


In [27]:
best_result = sorted(results, key=lambda r: (-r["acc"], r["loss"]))[0]

W1 = best_result["W1"]
b1 = best_result["b1"]
W2 = best_result["W2"]
b2 = best_result["b2"]
W3 = best_result["W3"]
b3 = best_result["b3"]

losses = best_result["losses"]
y_pred_final = best_result["y_pred"]
acc_final = best_result["acc"]

print("\n=== Meilleur modèle ===")
print("Seed retenu      :", best_result["seed"])
print("Loss finale      :", round(best_result["loss"], 4))
print("Accuracy finale  :", f"{acc_final:.2%}")


=== Meilleur modèle ===
Seed retenu      : 1
Loss finale      : 0.0506
Accuracy finale  : 100.00%


In [28]:
h = 0.02
xx, yy = np.meshgrid(
    np.arange(X[:, 0].min() - 0.2, X[:, 0].max() + 0.2, h),
    np.arange(X[:, 1].min() - 0.2, X[:, 1].max() + 0.2, h)
)
grid = np.c_[xx.ravel(), yy.ravel()]

a1g = tanh(grid @ W1 + b1)
a2g = tanh(a1g @ W2 + b2)
zg = sigmoid(a2g @ W3 + b3).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].contourf(xx, yy, zg, alpha=0.4, cmap="RdBu")
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", s=10, edgecolors="none")
axes[0].set_title("Frontière de décision (meilleur modèle)")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

axes[1].plot(losses)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss BCE")
axes[1].set_title("Courbe de loss spirale")

plt.savefig("phase4_spirale.png", dpi=100, bbox_inches="tight")
plt.show()

print("Figure sauvegardée : phase4_spirale.png")

Figure sauvegardée : phase4_spirale.png


C:\Users\serge\AppData\Local\Temp\ipykernel_13208\3965392058.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Vérification attendue et observations

### Scénario normal
On doit observer :
- une accuracy finale > 90% ;
- une loss qui descend nettement ;
- une frontière de décision qui suit les spirales ;
- une image exportée : `phase4_spirale.png`.

### Cas limite
Si on remplace l’architecture par 2–2–1, le réseau sous-fitte :
- accuracy plus faible ;
- frontière plus grossière ;
- capacité insuffisante pour modéliser correctement la spirale.

### Scénario adversarial
Si on augmente le bruit à `noise = 0.5`, la frontière devient plus irrégulière et l’accuracy baisse.
On peut alors comparer :
- `noise = 0.15`
- `noise = 0.5`

### Conclusion
- la spirale est plus difficile que XOR ;
- un réseau plus capable apprend une frontière plus fine ;
- un réseau trop petit sous-fitte visiblement ;
- le bruit rend la tâche plus difficile.